# UPI Rakshak AI — Multilingual Transformer Fine-Tuning
### Google Colab Free T4 GPU Accelerated Training

This notebook trains a specialized Indic Transformer (**Google MuRIL** or **AI4Bharat IndicBERT**) for 15-class digital payment scam detection across English, Hindi, Telugu, and Hinglish.

**Hardware**: Select `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repository or upload synthetic_scam_dataset.csv
# !git clone https://github.com/your-username/upi-rakshak-ai.git
# %cd upi-rakshak-ai

import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

MODEL_NAME = "google/muril-base-cased"  # Alternative: "ai4bharat/indic-bert"
OUTPUT_DIR = "./indic_transformer_model"

# Load dataset
df = pd.read_csv("backend/app/data/synthetic_scam_dataset.csv")
df = df.drop_duplicates(subset=["text", "category"]).reset_index(drop=True)
categories = sorted(df["category"].unique())
cat2id = {c: i for i, c in enumerate(categories)}
id2cat = {i: c for i, c in enumerate(categories)}

print(f"Dataset: {len(df)} samples across {len(categories)} categories.")

In [ ]:
# Stratified Group Split by template_id (Zero Leakage)
rng = np.random.RandomState(42)
train_idx, val_idx = [], []
for cat in categories:
    cdf = df[df["category"] == cat]
    tpls = list(cdf["template_id"].unique())
    rng.shuffle(tpls)
    split = max(1, int(len(tpls) * 0.8))
    tr_t, va_t = set(tpls[:split]), set(tpls[split:]) or set(tpls)
    train_idx.extend(cdf[cdf["template_id"].isin(tr_t)].index)
    val_idx.extend(cdf[cdf["template_id"].isin(va_t)].index)

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_fn(b): return tokenizer(b["text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df).map(lambda x: {"label": cat2id[x["category"]]}).map(tokenize_fn, batched=True)
val_ds = Dataset.from_pandas(val_df).map(lambda x: {"label": cat2id[x["category"]]}).map(tokenize_fn, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(categories),
    id2label=id2cat,
    label2id=cat2id
)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, tokenizer=tokenizer)
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"Fine-tuned model saved to {OUTPUT_DIR}")